In [2]:
import pandas as pd
import ast
from sklearn.feature_extraction.text import CountVectorizer

## Data Preparation

In [3]:
# Load Person A's cleaned dataset
df = pd.read_csv("Case4CleanedTeePublic.csv")

# Check the dataset
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Dataset shape: (200, 13)
Columns: ['reviewer_id', 'store_location', 'latitude', 'longitude', 'date', 'month', 'year', 'title', 'review_text', 'star_rating', 'clean_text', 'topic_tokens', 'text_source']


,reviewer_id,store_location,latitude,longitude,date,month,year,title,review_text,star_rating,clean_text,topic_tokens,text_source
0,2516,US,37.09024,-95.712891,2023,6,2008 00:00:00,Everyiwas very helpful,Great job problem solving,5,great job problem solving,"[""great"", ""job"", ""problem"", ""solving""]",review_body
1,3102,US,37.09024,-95.712891,2023,6,2006 00:00:00,Constant Updating of Delivery,Constant Updating of Delivery information,4,constant updating of delivery information,"[""constant"", ""updating"", ""delivery"", ""informat...",review_body
2,8694,US,37.09024,-95.712891,2023,5,2013 00:00:00,On time delivery,On time delivery. No hassle ordering.,5,on time delivery no hassle ordering,"[""time"", ""delivery"", ""no"", ""hassle"", ""ordering""]",review_body
3,9314,US,37.09024,-95.712891,2023,5,2005 00:00:00,Easy to order,Easy to order.Great shipping.,5,easy to order great shipping,"[""easy"", ""order"", ""great"", ""shipping""]",review_body
4,17105,US,37.09024,-95.712891,2023,3,2028 00:00:00,I love it it fits perfectly and also my,I love it it fits perfectly and also my favori...,5,i love it it fits perfectly and also my favori...,"[""love"", ""fits"", ""perfectly"", ""favorite"", ""mov...",review_body


In [4]:
# Convert stored token strings into Python lists
df["topic_tokens"] = df["topic_tokens"].apply(ast.literal_eval)

# Check the first few tokenized reviews
df[["review_text", "topic_tokens"]].head()

,review_text,topic_tokens
0,Great job problem solving,"[great, job, problem, solving]"
1,Constant Updating of Delivery information,"[constant, updating, delivery, information]"
2,On time delivery. No hassle ordering.,"[time, delivery, no, hassle, ordering]"
3,Easy to order.Great shipping.,"[easy, order, great, shipping]"
4,I love it it fits perfectly and also my favori...,"[love, fits, perfectly, favorite, movie, thank..."


In [5]:
# Convert token lists back into space-separated text
documents = df["topic_tokens"].apply(lambda tokens: " ".join(tokens))

# Create a document-term matrix
vectorizer = CountVectorizer(
    max_features=1000,
    min_df=2,
    max_df=0.95
)

dtm = vectorizer.fit_transform(documents)

# Display matrix information
print("Document-term matrix shape:", dtm.shape)
print("Vocabulary size:", len(vectorizer.get_feature_names_out()))

Document-term matrix shape: (200, 347)
Vocabulary size: 347


## Fit 2 LDA Models

In [6]:
from sklearn.decomposition import LatentDirichletAllocation

In [7]:
# LDA model with 5 topics
lda_5 = LatentDirichletAllocation(
    n_components=5,
    random_state=42,
    max_iter=20
)

lda_5.fit(dtm)

# LDA model with 10 topics
lda_10 = LatentDirichletAllocation(
    n_components=10,
    random_state=42,
    max_iter=20
)

lda_10.fit(dtm);

In [8]:
# Function to display the top words for each topic
def display_topics(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[-n_top_words:][::-1]
        top_words = [feature_names[i] for i in top_indices]

        print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")


# Get vocabulary words
feature_names = vectorizer.get_feature_names_out()

# Display topics for k = 5
print("LDA Model: 5 Topics")
display_topics(lda_5, feature_names)

# Display topics for k = 10
print("\nLDA Model: 10 Topics")
display_topics(lda_10, feature_names)

LDA Model: 5 Topics
Topic 1: service, customer, not, received, shirt, order, size, great, shirts, ordered
Topic 2: great, shirt, very, quality, not, ordered, fast, shirts, love, christmas
Topic 3: shirts, not, great, shirt, ordered, quality, good, like, fit, got
Topic 4: delivery, quality, love, quick, super, shirts, order, good, designs, fast
Topic 5: shirts, well, good, shirt, quality, very, designs, fit, fast, excellent

LDA Model: 10 Topics
Topic 1: size, shirts, ordered, no, shirt, not, service, customer, fit, get
Topic 2: shirt, great, fast, delivery, quality, very, teepublic, christmas, came, super
Topic 3: shirts, not, got, great, shipping, ordered, long, order, time, love
Topic 4: super, shirts, customer, service, great, perfect, delivered, expected, issue, received
Topic 5: very, good, quality, fast, shirt, prices, pleased, purchase, satisfied, product
Topic 6: great, shirts, shirt, quality, well, order, service, love, ordered, very
Topic 7: shirts, not, like, order, good, te

## Calculate Coherence Scores

In [9]:
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary

In [10]:
# Get tokenized reviews
texts = df["topic_tokens"].tolist()

# Create Gensim dictionary
dictionary = Dictionary(texts)

# Get vocabulary from CountVectorizer
feature_names = vectorizer.get_feature_names_out()

# Function to calculate coherence
def calculate_coherence(model, feature_names, texts, dictionary):
    topics = []

    for topic in model.components_:
        top_indices = topic.argsort()[-10:][::-1]
        top_words = [feature_names[i] for i in top_indices]
        topics.append(top_words)

    coherence_model = CoherenceModel(
        topics=topics,
        texts=texts,
        dictionary=dictionary,
        coherence="c_v"
    )

    return coherence_model.get_coherence()


# Calculate coherence for both LDA models
coherence_5 = calculate_coherence(
    lda_5, feature_names, texts, dictionary
)

coherence_10 = calculate_coherence(
    lda_10, feature_names, texts, dictionary
)

# Display results
print(f"LDA (5 topics) coherence: {coherence_5:.4f}")
print(f"LDA (10 topics) coherence: {coherence_10:.4f}")

LDA (5 topics) coherence: 0.3514
LDA (10 topics) coherence: 0.4285


The LDA model with 10 topics achieved a higher C_v coherence score (0.4285) than the model with 5 topics (0.3514). This suggests that the 10-topic model produced more semantically coherent topics so we are selecting the 10-topic model for further interpretation and category selection.

## Category Selection

We selected Sizing and Fit as our classification category based on the results of the 10-topic LDA model. Topics 1 and 8 contained relevant terms such as "size," "sizing," "fit," and "large," indicating that clothing fit was a recurring theme in the reviews. This category is relevant to TeePublic because identifying reviews that discuss sizing can help the business monitor customer experiences and identify potential sizing inconsistencies.

## Create Seed Vocabulary

In [11]:
# Seed vocabulary for Sizing and Fit
seed_words = [
    "size",
    "sizing",
    "fit",
    "fits",
    "fitting",
    "small",
    "large",
    "tight",
    "loose",
    "medium",
    "xl",
    "xxl",
    "oversized",
    "undersized",
    "too small",
    "too big",
    "true to size",
    "runs small",
    "smaller",
    "larger",
    "big",
    "bigger"
]

print("Number of seed terms:", len(seed_words))
print("Seed vocabulary:", seed_words)

Number of seed terms: 22
Seed vocabulary: ['size', 'sizing', 'fit', 'fits', 'fitting', 'small', 'large', 'tight', 'loose', 'medium', 'xl', 'xxl', 'oversized', 'undersized', 'too small', 'too big', 'true to size', 'runs small', 'smaller', 'larger', 'big', 'bigger']


In [12]:
import re

# Count how many reviews contain each seed term
seed_counts = {}

for seed in seed_words:
    count = df["clean_text"].str.contains(
        r"\b" + re.escape(seed) + r"\b",
        case=False,
        regex=True
    ).sum()

    seed_counts[seed] = count

# Display frequencies
seed_counts_df = pd.DataFrame(
    seed_counts.items(),
    columns=["Seed Term", "Review Count"]
).sort_values("Review Count", ascending=False)

seed_counts_df

,Seed Term,Review Count
2,fit,22
0,size,18
1,sizing,9
3,fits,7
5,small,6
6,large,5
20,big,4
9,medium,4
10,xl,4
14,too small,3


## Weak Label Construction

In [13]:
import re

# Create a function to assign weak labels
def assign_weak_label(text):
    for seed in seed_words:
        pattern = r"\b" + re.escape(seed) + r"\b"

        if re.search(pattern, text, flags=re.IGNORECASE):
            return 1

    return 0


# Apply weak labeling to all reviews
df["weak_label"] = df["clean_text"].apply(assign_weak_label)

# Display label distribution
print("Weak label distribution:")
print(df["weak_label"].value_counts())

# Preview labeled reviews
df[["review_text", "weak_label"]].head(10)

Weak label distribution:
weak_label
0    148
1     52
Name: count, dtype: int64


,review_text,weak_label
0,Great job problem solving,0
1,Constant Updating of Delivery information,0
2,On time delivery. No hassle ordering.,0
3,Easy to order.Great shipping.,0
4,I love it it fits perfectly and also my favori...,1
5,"I ordered a medium, which is the size I wear f...",1
6,The tshirt is not true to size. Normal size is...,1
7,Designs were nice but sizing was way off. Abou...,1
8,What great company. Customer service was hassl...,0
9,Easy peasy! Great quality shirts!,0


## Manually Review

In [14]:
# Select all 52 positive reviews
positive_reviews = df[df["weak_label"] == 1]

# Randomly select 48 negative reviews
negative_reviews = df[df["weak_label"] == 0].sample(
    n=48,
    random_state=42
)

# Combine and shuffle the selected reviews
manual_sample = pd.concat(
    [positive_reviews, negative_reviews]
).sample(frac=1, random_state=42)

# Create a column for manually verified labels
manual_sample = manual_sample.copy()
manual_sample["manual_label"] = manual_sample["weak_label"]

# Display all 100 reviews
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)

manual_sample[["review_text", "weak_label", "manual_label"]]

,review_text,weak_label,manual_label
65,"Amazing shirt design, love it!",0,0
71,I received the wrong order! I ordered a shirt for my mother for mothers day instead received a shirt that had a curse work on it. My receipt show the mothers day shirt. I emailed customer support and did not get a response! Horrible.,0,0
172,Excellent quality t-shirt. Arrived quickly. Thanks.,0,0
165,"I'll admit I was nervous about what to expect in terms of quality/shipment.. It all seemed a little too good to be true. Fortunately, I'm am so pleased with my purchase!! Email communication was great, and my shipment arrived without issue swiftly. On top of that, The Crewnecks I ordered look amazing. I'm not worried about the design fading or cracking, the size is wonderful - spot on, and the quality is amazing. The sweaters are soft and warm while being light. I am very excited to order more!",1,1
164,Loved it and fit great.,1,1
150,"This is my second time purchasing a large selectin of t-shirts. I am in Canada. For some reason this package arrived with significant customs charges which surprised me. Just be aware. Also I found the sizing for women and youth (age 12-18) very difficult. I have quite a few shirts which won't fit and I'll have to donate somewhere. Please add to the size range photos you have on your site. The adults men are a great range ( I got every singe one of them correct in the last order) and very easy to figure out. The women and youth, not so much. I sure enjoy the quality of graphics and the selection.",1,1
81,"Image print quality is good, although the color is more of a faded red rather than the brighter, truer red as it is depicted on the site, but my son likes it so it's fine. Fits him well, too.I do like the type of sizing charts on the site, it really does help to get a better idea of how each size will fit on different body shapes and sizes. Hoodie was printed and shipped out quickly.",1,1
113,Excellent transaction accurate shipping estimate great selection,0,0
32,Teepublic is incredible! I ordered 2 hoodies. But they just didn't fit quite right.So I contacted customer service and I offered to exchange the items and pay the shipping.They said that wouldn't be necessary.They would send me the size changes I requested at no additional charge. And that I should KEEP OR DONATE THE ORIGINAL ORDER!!! I was blown away.I have found two wonderful friends that will fit into those items nicely.And I've now received my replacement articles and they fit perfectly!! One last note.My items came in water proof packaging which I'm very grateful for. It was raining hard the day they were delivered and my items would have been soaked otherwise.This company rocks.I will be recommending them every opportunity that I get and I will certainly be buying from them again soon. Quality Customer service is incredibly important to me. Long term customer right here!!!!!,1,1
4,I love it it fits perfectly and also my favorite movie thank you guys great work,1,1


## Correct Labels

In [15]:
# Correct labels identified during manual validation
manual_sample.loc[26, "manual_label"] = 0
manual_sample.loc[44, "manual_label"] = 1
manual_sample.loc[55, "manual_label"] = 0
manual_sample.loc[104, "manual_label"] = 1
manual_sample.loc[144, "manual_label"] = 0

# Display corrected reviews
manual_sample.loc[
    [26, 44, 55, 104, 144],
    ["review_text", "weak_label", "manual_label"]
]

,review_text,weak_label,manual_label
26,T-shirts look good and they arrived even quicker than I expected (which was a big plus).,1,0
44,"Every year I give my husband a shirt or sweater that he ""can"" wear on Christmas.He will love the long sleeve tee shirt that I will give him this year!His favorite song of all time is Radar Love and I got the shirt in red.It's a nice weight tee shirt and extra roomy. Very fast delivery.",0,1
55,One shirt came with a big bright red stain and the replacement shirt came with a very crooked picture. Never again will I order from this site. Very careless work on behalf of whoever creates these shirts.,1,0
104,"The quality of shirts are always great, love the selection, and the customer service when sizes/prints weren't quite right was amazing and prompt! So easy I have made a few orders already. Cheers!",0,1
144,banginthought it might be low quality or something but both my shirts with pretty complicated designs with small words actually came out perfect,1,0


During manual validation, we identified five reviews that were incorrectly labeled by the seed-based rules:

- **Review 26 (1 → 0):** The phrase "big plus" refers to fast delivery rather than clothing size or fit.
- **Review 44 (0 → 1):** The phrase "extra roomy" describes the fit of the shirt, even though "roomy" was not included in our seed vocabulary.
- **Review 55 (1 → 0):** The word "big" describes a stain on the shirt rather than its size.
- **Review 104 (0 → 1):** The review mentions issues with "sizes," but our seed vocabulary only included "size," so the rule did not detect it.
- **Review 144 (1 → 0):** The phrase "small words" refers to the printed design rather than the size of the shirt.

In [16]:
# Compare weak labels with manually verified labels
corrections = (
    manual_sample["weak_label"] != manual_sample["manual_label"]
).sum()

total_checked = len(manual_sample)

print("Reviews manually checked:", total_checked)
print("Incorrect weak labels:", corrections)
print("Correct weak labels:", total_checked - corrections)
print("Agreement rate:", round(
    (total_checked - corrections) / total_checked * 100, 2
), "%")

Reviews manually checked: 100
Incorrect weak labels: 5
Correct weak labels: 95
Agreement rate: 95.0 %


## Merge Manually Verified Labels

In [17]:
# Start with the weak labels for all 200 reviews
df["final_label"] = df["weak_label"].copy()

# Replace weak labels with manually verified labels
df.loc[manual_sample.index, "final_label"] = manual_sample["manual_label"]

# Identify which reviews were manually checked
df["manually_checked"] = df.index.isin(manual_sample.index)

# Display final label distribution
print("Final label distribution:")
print(df["final_label"].value_counts())

# Display number of manually checked reviews
print("\nManually checked reviews:", df["manually_checked"].sum())

# Preview final dataset
df[["review_text", "weak_label", "final_label", "manually_checked"]].head(10)

Final label distribution:
final_label
0    149
1     51
Name: count, dtype: int64

Manually checked reviews: 100


,review_text,weak_label,final_label,manually_checked
0,Great job problem solving,0,0,False
1,Constant Updating of Delivery information,0,0,False
2,On time delivery. No hassle ordering.,0,0,False
3,Easy to order.Great shipping.,0,0,False
4,I love it it fits perfectly and also my favorite movie thank you guys great work,1,1,True
5,"I ordered a medium, which is the size I wear for other T shirts, and it is so small.",1,1,True
6,The tshirt is not true to size. Normal size is a large womens. Originally ordered a large and it was tight. Then I was able to get an replacement with no return fees or hassles and itthe xl is still too small. That means I would have to get a mens tshirt or XXL in woman's .I don't believe the company wou,1,1,True
7,Designs were nice but sizing was way off. About two sizes too small from what was expected.,1,1,True
8,What great company. Customer service was hassle free. They had another order on the way and said no worries.,0,0,True
9,Easy peasy! Great quality shirts!,0,0,False


In [18]:
# Save the final labeled dataset
df.to_csv("Case4_Labeled.csv", index=False)

In [19]:
# Reload the exported dataset
labeled_df = pd.read_csv("Case4_Labeled.csv")

print("Dataset shape:", labeled_df.shape)

print("\nFinal label distribution:")
print(labeled_df["final_label"].value_counts())

print("\nManually checked reviews:")
print(labeled_df["manually_checked"].value_counts())

Dataset shape: (200, 16)

Final label distribution:
final_label
0    149
1     51
Name: count, dtype: int64

Manually checked reviews:
manually_checked
False    100
True     100
Name: count, dtype: int64


## Topic Table and Label Construction Methodology

In [20]:
# Function to create a topic summary table
def create_topic_table(model, feature_names, n_topics):
    topic_data = []

    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[-10:][::-1]
        top_words = [feature_names[i] for i in top_indices]

        topic_data.append({
            "Model": f"k={n_topics}",
            "Topic": f"Topic {topic_idx + 1}",
            "Top 10 Words": ", ".join(top_words)
        })

    return pd.DataFrame(topic_data)


# Create tables for both models
topics_5 = create_topic_table(lda_5, feature_names, 5)
topics_10 = create_topic_table(lda_10, feature_names, 10)

# Combine both tables
topic_table = pd.concat([topics_5, topics_10], ignore_index=True)

# Add coherence scores
topic_table["Coherence"] = topic_table["Model"].map({
    "k=5": round(coherence_5, 4),
    "k=10": round(coherence_10, 4)
})

# Display final topic table
pd.set_option("display.max_colwidth", None)
topic_table

,Model,Topic,Top 10 Words,Coherence
0,k=5,Topic 1,"service, customer, not, received, shirt, order, size, great, shirts, ordered",0.3514
1,k=5,Topic 2,"great, shirt, very, quality, not, ordered, fast, shirts, love, christmas",0.3514
2,k=5,Topic 3,"shirts, not, great, shirt, ordered, quality, good, like, fit, got",0.3514
3,k=5,Topic 4,"delivery, quality, love, quick, super, shirts, order, good, designs, fast",0.3514
4,k=5,Topic 5,"shirts, well, good, shirt, quality, very, designs, fit, fast, excellent",0.3514
5,k=10,Topic 1,"size, shirts, ordered, no, shirt, not, service, customer, fit, get",0.4285
6,k=10,Topic 2,"shirt, great, fast, delivery, quality, very, teepublic, christmas, came, super",0.4285
7,k=10,Topic 3,"shirts, not, got, great, shipping, ordered, long, order, time, love",0.4285
8,k=10,Topic 4,"super, shirts, customer, service, great, perfect, delivered, expected, issue, received",0.4285
9,k=10,Topic 5,"very, good, quality, fast, shirt, prices, pleased, purchase, satisfied, product",0.4285


We fitted two LDA models with 5 and 10 topics to explore recurring themes in the reviews. The 10-topic model achieved a higher C_v coherence score (0.4285) than the 5-topic model (0.3514), suggesting that it produced more semantically coherent topics.

We selected Sizing and Fit as our classification category based on Topics 1 and 8 of the 10-topic model. These topics contained relevant words such as "size," "sizing," "fit," and "large." This category is relevant to TeePublic because identifying reviews that discuss clothing fit can help the business monitor sizing concerns and identify potential inconsistencies.

Using the LDA results and domain knowledge, we developed a 22-term seed vocabulary to identify reviews discussing Sizing and Fit.

We used a hybrid labeling approach combining seed-based weak labeling with manual validation. Using our 22-term seed vocabulary for Sizing and Fit, we assigned provisional labels to all 200 reviews. Reviews containing at least one seed term were labeled 1 (category present), while reviews without matching terms were labeled 0 (category absent).

We then manually reviewed 100 reviews, including all 52 provisionally positive reviews and 48 randomly selected negative reviews. During validation, we identified and corrected five incorrectly labeled reviews, resulting in a 95% agreement rate between the weak labels and manual judgments. The errors included false positives where keywords appeared in unrelated contexts and false negatives where sizing was discussed using words outside our seed vocabulary.

After incorporating these corrections, our final dataset contained 51 positive reviews (25.5%) and 149 negative reviews (74.5%). Although this approach allowed us to efficiently label the dataset, the remaining 100 unverified reviews may still contain labeling errors.

In [23]:
import dataframe_image as dfi

dfi.export(
    topic_table,
    "Case4_LDA_Topic_Table.png",
    table_conversion="matplotlib"
)

print("Table saved successfully!")

Table saved successfully!
